# W03 · 01 탄소 원자쌍 한 스텝 손계산 재현 → Δt 스윕
목표 3
1. 덱 슬라이드 12 · 23 의 숫자(U −0.102 · F −0.0606 · a −2.11e−6 · r 4.00 → 4.01 Å)를 코드로 재현
2. velocity-Verlet 한 스텝을 식 그대로 작성
3. C–C 결합 1차원 진동에서 Δt 0.5 ~ 8 fs 의 에너지 오차 표 → 슬라이드 25 와 같은 값

단위: LAMMPS real (Å · fs · kcal/mol · g/mol) · 환산 1 (kcal/mol/Å)/(g/mol) = 4.184 × 10⁻⁴ Å/fs²

실행(기본): week03 폴더에서 연 Claude Code 에 「`@notebooks/01_lj_pair_verlet.ipynb` 를 실행하고 결과를 요약해」 → 승인 → `python ../run_nb.py notebooks/01_lj_pair_verlet.ipynb` · 그림을 직접 보려면 `jupyter lab`

In [ ]:
# 기본 = 내 컴퓨터 mdai 환경 · Claude Code 가 week03 폴더에서 실행 · 아래 google.colab 분기는 Colab(선택)에서만 동작
import os, sys
if "google.colab" in sys.modules and not os.path.exists("scripts/lj_verlet.py"):
    !git clone https://github.com/sentirain7/2026_MOOC_Lecture_Materials.git repo && %cd repo/week03
import numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, "scripts")
from lj_verlet import *   # EPS SIG M CONV KB lj_u lj_f pair_step bond_sweep lj_fluid OUT
print("ε", EPS, "σ", SIG, "m", M)

## 1. 퍼텐셜 · 힘 · 가속도 (슬라이드 12 ①②③)

In [ ]:
r = 4.00
print("U(r)  =", round(lj_u(r), 4), "kcal/mol        (덱: −0.102)")
print("F(r)  =", round(lj_f(r), 4), "kcal/mol·Å      (덱: −0.0606, 음수 = 인력)")
print("a     =", f"{lj_f(r)/M*CONV:.3e}", "Å/fs²      (덱: −2.11e−6)")
rmin = 2**(1/6)*SIG; print("r_min =", round(rmin, 3), "Å · U(r_min) =", round(lj_u(rmin), 4), "= −ε")

## 2. velocity-Verlet 한 스텝 (슬라이드 23 순서 ①~⑤)

In [ ]:
def verlet_step(r, v, dt):
    a = lj_f(r) / M * CONV            # ① 힘
    v = v + 0.5 * a * dt              # ② 속도 반스텝
    r = r + v * dt                    # ③ 좌표 전진
    a = lj_f(r) / M * CONV            # ④ 힘 재계산
    v = v + 0.5 * a * dt              # ⑤ 속도 나머지 반스텝
    return r, v
r1, v1 = verlet_step(4.00, 0.0100, 1.0)
print(f"r 4.0000 → {r1:.8f} Å · v 0.0100 → {v1:.8f} Å/fs   (덱 23장 노트: 4.00999894 · 0.00999786)")
print(json.dumps(pair_step(), indent=1, ensure_ascii=False)) if (json:=__import__("json")) else None

## 3. Δt 스윕 — C–C 결합 1차원 진동 (슬라이드 25 와 같은 모델)
U = K(r − r₀)² · K 478.4 kcal/mol·Å² · r₀ 1.387 Å (GAFF2 방향족 C–C) · 환산질량 μ = 12.011/2
이론 주기 T = 2π/ω · velocity-Verlet 안정 조건 ωΔt < 2

In [ ]:
sw = bond_sweep(dts=(0.5, 1, 2, 4, 6, 8))
print("period", sw["period_fs"], "fs · stability dt <", sw["stability_dt_fs"], "fs")
for row in sw["rows"]:
    e = "발산" if row["diverged"] else f'{row["max_rel_energy_error"]*100:.2f} %'
    print(f'Δt {row["dt_fs"]:>4} fs  ωΔt {row["omega_dt"]:.3f}  max|ΔE|/E₀ {e}')

In [ ]:
# 에너지 시계열 그림 (Δt 1 · 4 · 8 fs)
K, r0, mu = 478.4, 1.387, M/2
fig, ax = plt.subplots(figsize=(8, 3.5))
for dt, c in [(1, "tab:blue"), (4, "tab:orange"), (8, "tab:red")]:
    x, v = 0.10, 0.0; a = -2*K*x*CONV/mu; E0 = K*x*x; ts, es = [], []
    for i in range(int(200/dt)):
        v += 0.5*a*dt; x += v*dt; a = -2*K*x*CONV/mu; v += 0.5*a*dt
        ts.append(i*dt); es.append((0.5*mu*v*v/CONV + K*x*x)/E0)
        if abs(x) > 5: break
    ax.plot(ts, es, color=c, label=f"dt = {dt} fs")
ax.set_ylim(0, 2); ax.axhline(1, ls="--", color="gray"); ax.set_xlabel("t (fs)"); ax.set_ylabel("E / E0"); ax.legend()
fig.tight_layout(); fig.savefig(OUT / "dt_sweep.png", dpi=120); plt.show()

## 확인 질문 (quiz.md 참조)
1. 한 스텝에서 속도를 두 번에 나눠 갱신하는 이유
2. Δt 를 절반으로 줄이면 한 스텝 국소 오차는 몇 분의 1 인가
3. 위 표에서 Δt 8 fs 가 발산하는 조건을 ωΔt 로 쓰면